# RAG pipeline

In [8]:
import os
import sys
import csv
import ast
import json
import numpy as np
import pandas as pd
from ast import literal_eval
from dotenv import load_dotenv
from llama_index.core import (
    Document,
    VectorStoreIndex,
    StorageContext,
    PromptTemplate,
    load_index_from_storage,
    set_global_handler
)
from llama_index.llms.vllm import Vllm
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.settings import Settings
from llama_cloud import AsyncLlamaCloud

# logging
import logging
logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("llama_index.core").setLevel(logging.WARNING)
logging.getLogger("fsspec").setLevel(logging.WARNING)
set_global_handler("simple")

load_dotenv()
CACHE_DIR = "../parsed_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

client = AsyncLlamaCloud(api_key=os.getenv("LLAMA_CLOUD_API_KEY"))

## Setup LLM and Embedding model

* For the LLM, use **Gemini** for local development and *vLLM* for production
* For the embedding model use **BAAI/bge-small-en-v1.5** for local and Qwen3

In [23]:
if os.getenv("APP_ENV") == "dev":
    embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
    llm = GoogleGenAI(model="gemini-2.5-flash-lite")
elif os.getenv("APP_ENV") == "prod":
    embed_model = HuggingFaceEmbedding(model_name="Qwen/Qwen3-Embedding-0.6B")
    llm = Vllm(
        model="Qwen3.5-122B-A10B-GGUF",
        tensor_parallel_size=4,
        max_new_tokens=100,
        vllm_kwargs={"swap_space": 1, "gpu_memory_utilization": 0.5},
    )

Settings.llm = llm
Settings.embed_model = embed_model

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2486.84it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


INFO:sentence_transformers.SentenceTransformer:1 prompt is loaded, with the key: query


# Parse the PDF

In [24]:
async def parse_documents_with_llamaparse(data_dir: str):
    documents = []

    for filename in os.listdir(data_dir):
        if not filename.endswith(".pdf"):
            continue

        cache_file = os.path.join(CACHE_DIR, f"{filename}.json")

        # load from cache
        if os.path.exists(cache_file):
            print(f"Loading cached parse for {filename}...")
            with open(cache_file, "r") as f:
                pages = json.load(f)

            for page in pages:
                documents.append(
                    Document(
                        text=page["text"],
                        metadata=page["metadata"]
                    )
                )
            continue

        # parse normally
        file_path = os.path.join(data_dir, filename)

        print(f"Uploading {filename} to LlamaCloud...")
        file_obj = await client.files.create(
            file=file_path,
            purpose="parse"
        )

        print("Parsing file...")
        result = await client.parsing.parse(
            file_id=file_obj.id,
            tier="cost_effective",
            version="latest",
            agentic_options={
                "custom_prompt": "This is an equipment manual..."
            },
            output_options={
                "markdown": {
                    "tables": {"output_tables_as_markdown": True},
                },
                "images_to_save": ["embedded"],
            },
            expand=["markdown"]
        )

        print("Saving to cache...")
        pages_to_save = []
        for page in result.markdown.pages:
            pages_to_save.append({
                "text": page.markdown,
                "metadata": {
                    "file_name": filename,
                    "page": page.page_number
                }
            })

            documents.append(
                Document(
                    text=page.markdown,
                    metadata={
                        "file_name": filename,
                        "page": page.page_number
                    }
                )
            )

        with open(cache_file, "w") as f:
            json.dump(pages_to_save, f)

    return documents

documents = await parse_documents_with_llamaparse("../data")

Loading cached parse for Getting_Started_with_SM_Datum_V7.2.4_2024_CT_EN.pdf...
Loading cached parse for User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN.pdf...


## Chunk the document and store the RAG index

NOTE: If the parsing or chunking strategy is changed it will NOT automatically update your existing stored index. It loads the old index with old chunks
* After applying a new parsing or chunking strategy, delete the old storage folder.

In [ ]:
# chunking
splitter = SentenceSplitter(
    chunk_size=512,
    chunk_overlap=50
)

print("Chunking documents into nodes...")
nodes = splitter.get_nodes_from_documents(documents)

# vector index
if os.path.exists("../storage"):
    print("Loading index from storage...")
    storage_context = StorageContext.from_defaults(persist_dir="../storage")
    index = load_index_from_storage(storage_context)
else:
    print("Creating new index...")
    index = VectorStoreIndex.from_documents(nodes)
    index.storage_context.persist("../storage")

Chunking documents into nodes...
Loading index from storage...


In [26]:
for node in nodes:
    print(f"{node.node_id}: {node.text}")

27c5aa17-cfc2-45ac-af02-5ad5c508f693: www.contrastech.com

# Getting Started with SM-Datum

![Icon of a camera lens](page_1_image_1_v2.jpg)

Ver 2.3.4 Mar. 2024

Vision System
a96bef05-3351-4a1e-b51b-20c3db8c0e98: ![Perfect](page_2_image_1_v2.jpg)

# Perfect

## Purpose of This Guide

This guide describes the function of the SM-Datum and how to use it and gives a detailed description of each tools. The information contained in the Manual is subject to change, without notice, due to firmware updates or other reasons. Please find the latest version of this Manual at the Contrastech website.Please read this guide before use.

Copyright ©2023
Hangzhou Contrastech Co., Ltd.
Tel: 86-571-89712238
Add.: No.8, Xiyuan 9th Road West Lake District, Hangzhou 310030 China

All rights reserved. The information contained herein is proprietary and is provided solely for the purpose of allowing customers to operate and/or service Contrastech manufactured equipment and is not to be released, reproduced, 

In [27]:
# basic retriever
retriever = index.as_retriever(similarity_top_k=5)

print("Querying the index...")
query_engine = RetrieverQueryEngine.from_args(
    retriever,
    llm=llm
)

print("Generating response...")
response = await query_engine.aquery(
    "What is the name of this device?"
)
print(response)

Querying the index...
Generating response...
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
** Messages: **
system: You are an expert Q&A system that is trusted around the world.
Always answer the query using the provided context information, and not prior knowledge.
Some rules to follow:
1. Never directly reference the given context in your answer.
2. Avoid statements like 'Based on the context, ...' or 'The context information ...' or anything along those lines.
user: Context information is below.
---------------------
file_name: User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN.pdf
page: 6

Equipment shown in Fig. 1-2

Double-click the SM-Datum shortcut on the desktop to open up the client software.

| Color  | Pin | Signal    | Signal Source | Designation                                                                    | Cable type               |
| ------ | --- | --------- | ------------- | -----------------------------------------------------------------------------

# RAG Evaluation

In [21]:
# # add a new question to the csv
# with open("../eval_dataset.csv", "a", newline="", encoding="utf-8") as csvfile:
#     writer = csv.writer(csvfile)
#     writer.writerow([
#         "",
#         "",
#         [""]
#     ])

In [17]:
eval_dataset = pd.read_csv("../eval_dataset.csv")
eval_dataset

,document,question,ground_truth,contexts
0,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,What is the order model number for the M12 cable?,['The order model number for the 17-pin M12 ca...,['### 17-pin M12 Cable with Fast Ethernet Inte...
1,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,What are the dimensions of the M12-mount housi...,['The dimensions of the M12-mount housing vari...,['Fig. 1-1: $46 \times 57.6 \times 25$ mm Mech...
2,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,What are all the pins on the 17-pin connector ...,['The pins on the 17-pin connector that should...,['| Purple White | 3 | - | - ...
3,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,What communication protocols does the SM2 seri...,['The SM2 series smart cameras supports Serial...,"['* Support Serial, TCP, UDP, FTP, Modbus and ..."
4,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,List all the LED indicators and what each colo...,"[""The LED indicators and their meanings are as...","[""| PWR Indicator | It is a power indicator. T..."
5,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,What are the step-by-step instructions to inst...,[''],['']
6,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,How do you add a remote camera in SM-Datum?,[''],['']
7,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,How does the wiring differ when connecting a P...,[''],['']
8,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,I want to use GPIO0 as an output. What needs t...,[''],['']
9,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,I want to connect 8 NPN photoelectric sensors ...,[''],['']


In [18]:
# convert stringified lists into real Python lists
eval_dataset["contexts"] = eval_dataset["contexts"].apply(ast.literal_eval)

# remove rows where contexts is empty or only contains empty strings
eval_dataset = eval_dataset[
    eval_dataset["contexts"].apply(
        lambda x: isinstance(x, list) and any(str(item).strip() for item in x)
    )
]

## Evaluate Retrieval

In [20]:
def normalize(text: str) -> str:
    """Light normalization: collapse whitespace, lowercase."""
    import re
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)  # collapse newlines, tabs, extra spaces
    return text.strip()

def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
    a, b = np.array(vec_a), np.array(vec_b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

def is_relevant(
    retrieved_chunk: str,
    gt_chunks: list[str],
    embed_model,
    threshold: float
) -> bool:
    """
    A retrieved chunk is relevant if its cosine similarity to ANY
    ground truth chunk exceeds the threshold.
    """
    retrieved_vec = embed_model.get_text_embedding(normalize(retrieved_chunk))
    for gt_chunk in gt_chunks:
        gt_vec = embed_model.get_text_embedding(normalize(gt_chunk))
        if cosine_similarity(retrieved_vec, gt_vec) >= threshold:
            return True
    return False

In [45]:
def evaluate_retrieval(
    retriever,
    eval_dataset: pd.DataFrame,
    embed_model,
    threshold: float = 0.80,
    k: int = 5,
) -> dict:
    """
    Evaluate retrieval using MRR@K and Recall@K.
    """
    results = []

    for _, row in eval_dataset.iterrows():
        question = row["question"]

        # contexts column may be stored as a string representation of a list
        gt_chunks = row["contexts"]
        if isinstance(gt_chunks, str):
            gt_chunks = literal_eval(gt_chunks)

        # retrieve top-K chunks
        retrieved_nodes = retriever.retrieve(question)
        retrieved_chunks = [node.get_content() for node in retrieved_nodes[:k]]

        # Binarize relevance for each retrieved chunk
        relevance_list = [
            1 if is_relevant(chunk, gt_chunks, embed_model, threshold) else 0
            for chunk in retrieved_chunks
        ]

        # MRR
        rr = 0.0
        for rank, rel in enumerate(relevance_list, start=1):
            if rel == 1:
                rr = 1.0 / rank
                break

        # Recall@K
        recall = 1.0 if any(relevance_list) else 0.0

        results.append({
            "question": question,
            "contexts": row["contexts"],
            "document": row["document"],
            "retrieved_chunks": retrieved_chunks,
            "gt_chunks": gt_chunks,
            "relevance_list": relevance_list,
            "reciprocal_rank": rr,
            "recall": recall,
            "num_relevant": sum(relevance_list),
        })

    # --- Aggregate ---
    mrr = float(np.mean([r["reciprocal_rank"] for r in results]))
    recall_at_k = float(np.mean([r["recall"] for r in results]))

    print(f"Overall MRR@{k}:     {mrr:.4f}")
    print(f"Overall Recall@{k}:  {recall_at_k:.4f}")

    summary = {
        f"MRR@{k}": round(mrr, 4),
        f"Recall@{k}": round(recall_at_k, 4),
        "threshold_used": threshold,
        "num_questions": len(results),
        "per_question": results,
    }

    return summary

In [46]:
results = evaluate_retrieval(
    retriever=retriever,
    eval_dataset=eval_dataset,
    embed_model=embed_model,
    threshold=0.80,
    k=5,
)

Overall MRR@5:     0.4286
Overall Recall@5:  0.4286


### Per-question analysis

In [34]:
per_q_df = pd.DataFrame(results["per_question"])
per_q_df.head()

,question,contexts,document,retrieved_chunks,gt_chunks,relevance_list,reciprocal_rank,recall,num_relevant
0,What is the order model number for the M12 cable?,[### 17-pin M12 Cable with Fast Ethernet Inter...,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,[| 8-pin terminal |\n\n17-pin M12 Ca...,[### 17-pin M12 Cable with Fast Ethernet Inter...,"[1, 0, 0, 0, 0]",1.0,1.0,1
1,What are the dimensions of the M12-mount housi...,[Fig. 1-1: $46 \times 57.6 \times 25$ mm Mecha...,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,[# 1 Mechanical Dimensions\n\nThe dimensions i...,[Fig. 1-1: $46 \times 57.6 \times 25$ mm Mecha...,"[1, 0, 0, 0, 0]",1.0,1.0,1
2,What are all the pins on the 17-pin connector ...,[| Purple White | 3 | - | - ...,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,[| Do not use it for wiring |\n| Purple | 13 ...,[| Purple White | 3 | - | - ...,"[0, 0, 0, 0, 0]",0.0,0.0,0
3,What communication protocols does the SM2 seri...,"[* Support Serial, TCP, UDP, FTP, Modbus and o...",User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,[Equipment shown in Fig. 1-2\n\nDouble-click t...,"[* Support Serial, TCP, UDP, FTP, Modbus and o...","[0, 0, 0, 0, 0]",0.0,0.0,0
4,List all the LED indicators and what each colo...,[| PWR Indicator | It is a power indicator. Th...,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,[# Status LED Description\n\n| Status LED |...,[| PWR Indicator | It is a power indicator. Th...,"[1, 0, 0, 0, 0]",1.0,1.0,1


### Failure cases

In [35]:
# Questions where nothing was retrieved correctly
failures = per_q_df[per_q_df["recall"] == 0.0]
print(f"\n{len(failures)} complete misses:\n")
for _, row in failures.iterrows():
    print(f"  [{row['document']}] {row['question']}")

# Questions where the first relevant chunk wasn't ranked #1
poor_mrr = per_q_df[(per_q_df["recall"] == 1.0) & (per_q_df["reciprocal_rank"] < 1.0)]
print(f"\n{len(poor_mrr)} retrieved but ranked poorly:\n")
for _, row in poor_mrr.iterrows():
    print(f"  RR={row['reciprocal_rank']:.2f} | {row['question']}")
    print(f"  Relevance list: {row['relevance_list']}")


4 complete misses:

  [User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN] What are all the pins on the 17-pin connector that should NOT be used for wiring?
  [User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN] What communication protocols does the SM2 series support?
  [User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN] Is the SM2 SE camera compatible with GigE Vision or GenICam standards?
  [User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN] Can the IO box V1.0 upgrade non-isolated IO to opto-isolated IO?

0 retrieved but ranked poorly:



In [37]:
def inspect_question(per_q_df, index):
    row = per_q_df.iloc[index]
    print(f"Question:     {row['question']}")
    print(f"Contexts: {row['contexts']}")
    print(f"RR: {row['reciprocal_rank']:.2f}  |  Recall: {row['recall']}")
    print(f"\nGT Chunks ({len(row['gt_chunks'])}):")
    for i, c in enumerate(row['gt_chunks']):
        print(f"  [{i}] {c[:200]}...")
    print(f"\nRetrieved Chunks (relevance: {row['relevance_list']}):")
    for i, (chunk, rel) in enumerate(zip(row['retrieved_chunks'], row['relevance_list'])):
        tag = "✓" if rel else "✗"
        print(f"  {tag} [{i}] {chunk[:200]}...")

inspect_question(per_q_df, 0)

Question:     What is the order model number for the M12 cable?
Contexts: ['### 17-pin M12 Cable with Fast Ethernet Interface\n\nOrder Model: VT-M1217P2RJ45-3M(SM2)\n\n']
RR: 1.00  |  Recall: 1.0

GT Chunks (1):
  [0] ### 17-pin M12 Cable with Fast Ethernet Interface

Order Model: VT-M1217P2RJ45-3M(SM2)

...

Retrieved Chunks (relevance: [1, 0, 0, 0, 0]):
  ✓ [0] | 8-pin terminal           |

17-pin M12 Cable with Fast Ethernet Interface 169.254.58.

Order Model: VT-M1217P2RJ45-3M(SM2)

Subnet Mask: 255.255.255.0

Default Gateway: 169.254.58.254

＊ The network...
  ✗ [1] I/O and serial ports

| Pin | Signal    | Signal Source | Designation                                                                    | Cable                    |
| --- | --------- | ------------- ...
  ✗ [2] | Do not use it for wiring |
| Green  | 4   | RS-232 TX | -             | RS-232 serial port output                                                      | DB9 female serial port   |
| Green  | 5   | R...
  ✗ [3

### Per-Document Analysis

In [44]:
doc_summary = per_q_df.groupby("document").agg(
    num_questions=("question", "count"),
    MRR=("reciprocal_rank", "mean"),
    Recall=("recall", "mean"),
).round(4)
print(doc_summary)

                                           num_questions     MRR  Recall
document                                                                
User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN              7  0.4286  0.4286


## Evaluate Generation

In [ ]:
judge_prompt = PromptTemplate(
    """You are evaluating an answer.

Question: {question}
Ground Truth: {ground_truth}
Answer: {answer}

Is the answer acceptable? Respond with ONLY 'yes' or 'no'.
"""
)

async def evaluate_with_llm(query_engine, eval_dataset, llm):
    scores = []

    for sample in eval_dataset:
        response = await query_engine.aquery(sample["question"])
        answer = str(response)

        judge_input = judge_prompt.format(
            question=sample["question"],
            ground_truth=sample["ground_truth"],
            answer=answer
        )

        judgment = await llm.acomplete(judge_input)
        is_correct = "yes" in judgment.text.lower()

        scores.append(is_correct)

    accuracy = sum(scores) / len(scores)
    print(f"LLM Judged Accuracy: {accuracy:.2f}")

await evaluate_with_llm(query_engine, eval_dataset, llm)

Retrieval Hit Rate: 1.00
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
** Messages: **
system: You are an expert Q&A system that is trusted around the world.
Always answer the query using the provided context information, and not prior knowledge.
Some rules to follow:
1. Never directly reference the given context in your answer.
2. Avoid statements like 'Based on the context, ...' or 'The context information ...' or anything along those lines.
user: Context information is below.
---------------------
file_name: User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN.pdf
page: 6

|
| Gray  | 6   | DI\_0     | It can be configured as input or output, and is input by default. |
| Black | 7   | GND       | DC Power Supply Negative                                          |
| Red   | 8   | POWER\_IN | DC Power Supply Positive                                          |

You cannot use DB9 female serial port and VCC to power the device at the same time. Otherwise, damaging to power sup